<center>

$\Huge \textbf{Universidad Nacional Autónoma de México}$  
$\Huge \textbf{Facultad de Ciencias}$  
<p align="center">
  <img src="https://www.icat.unam.mx/wp-content/uploads/2021/11/Copia-de-LogoUNAM.-Azul.-Fondo-transparente.png" alt="UNAM" width="200"/>
</p>

<hr style="height:3px; background-color:#0B6E4F; border:none;"/>


$\LARGE \textbf{Inteligencia Artificial}$  

$\Large \textit{Laboratorio 2.3}$  


\begin{array}{rl}
\textbf{Docente:} & Dra. Jessica Sarahi Méndez Rincón \\[6pt]
\textbf{Ayudante de laboratorio:} & Diego Eduardo Peña Villegas \\[6pt]
\textbf{Alumna:} & Marisol Luna Méndez \\[6pt]
\textbf{Fecha de realización:} & 25/02/2026
\end{array} 4

</center>

#Bibliotecas

In [1]:
# Importacion de librerias
import numpy as np
from numpy import genfromtxt
import matplotlib.pyplot as plt
import pandas as pd

# Spam

Usando la base de datos de spam (archivo spam.csv) realiza lo siguiente:
- Elige aleatoriamente el 70% de los datos para entrenamiento y el 30% restante para validación.
- Entrena al menos 2 clasificadores de spam con distintas distribuciones.

- Emplea los clasificadores entrenados para predecir spam tanto en los datos de entrenamiento como en los de validación y reporta el porcentaje de predicciones correctas de cada clasificador

- Discute el desempeño de los diferentes clasificadores

El archivo spam.csv contiene 2001 valores por cada renglón, de los cuales los primeros 2000 representan el histograma de palabras de un correo y el último corresponde a la clase, esto es, 1 si es spam y 0 si no lo es.

#Preparar los datos

In [2]:
# Importacion del archivo que se nos proporciono agregandole cabeceras que no contiene
names = ['correo' ]
dataset = pd.read_csv('spam.csv',names =names)
data=genfromtxt('spam.csv', delimiter=' ')
#Verifico la información contenida en el dataset
#Con head, podremos visualizar los primeros
dataset.head(5)

FileNotFoundError: [Errno 2] No such file or directory: 'spam.csv'

In [ ]:
# Asegúrate de que el delimitador sea el correcto (espacio, coma o tabulador)
dataset = pd.read_csv('spam.csv', sep='\s+', header=None)

# Verifica la forma: debería ser (N, 2001)
print(f"Forma del dataset: {dataset.shape}")

X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Suponiendo que 'dataset' ya cargó las 2001 columnas
X = dataset.iloc[:, :-1].values # Las primeras 2000 columnas
y = dataset.iloc[:, -1].values  # La última columna (clase)

# División 70% entrenamiento y 30% validación
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# Importante para SVM, KNN y Regresión Logística: Escalamiento
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

#2. Implementación de los 4 Clasificadores

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score


In [ ]:
# Diccionario para almacenar modelos
# Se usan modelos con distintas distribuciones:
#   - Logistic Regression: distribución logística
#   - BernoulliNB: distribución de Bernoulli (ideal para presencia/ausencia de palabras)
#   - KNN (k=5): basado en distancia, sin suposición de distribución
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes (Bernoulli)": BernoulliNB(),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5)
}


In [ ]:
# Entrenamiento y Evaluación
# NOTA: 'Accuracy' mide predicciones correctas / total.
#       'Precisión (Precision)' mide VP / (VP + FP).
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)

    train_acc  = accuracy_score(y_train, modelo.predict(X_train))
    test_acc   = accuracy_score(y_test,  modelo.predict(X_test))
    test_prec  = precision_score(y_test, modelo.predict(X_test), zero_division=0)

    print(f"{nombre}:")
    print(f"  Accuracy  (entrenamiento): {train_acc*100:.2f}%")
    print(f"  Accuracy  (validación):    {test_acc*100:.2f}%")
    print(f"  Precisión (validación):    {test_prec*100:.2f}%\n")


In [ ]:
nombres_graf = []
train_results = []
test_results  = []

for nombre, modelo in modelos.items():
    nombres_graf.append(nombre)
    train_results.append(accuracy_score(y_train, modelo.predict(X_train)) * 100)
    test_results.append(accuracy_score(y_test,  modelo.predict(X_test))  * 100)

x = np.arange(len(nombres_graf))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 7))
rects1 = ax.bar(x - width/2, train_results, width, label='Entrenamiento', color='#3498db')
rects2 = ax.bar(x + width/2, test_results,  width, label='Validación',    color='#e74c3c')

ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy de Clasificadores de Spam (Entrenamiento vs Validación)')
ax.set_xticks(x)
ax.set_xticklabels(nombres_graf)
ax.set_ylim(0, 115)
ax.legend()

def label_bars(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.1f}%',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom')

label_bars(rects1)
label_bars(rects2)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns

fig, axs = plt.subplots(1, 3, figsize=(15, 5))

for i, (nombre, modelo) in enumerate(modelos.items()):
    ConfusionMatrixDisplay.from_estimator(
        modelo,
        X_test,
        y_test,
        cmap='Greens',
        ax=axs[i],
        colorbar=False
    )
    axs[i].set_title(f'Matriz de Confusión\n{nombre}')

plt.tight_layout()
plt.show()


# Conclusión

En esta práctica se entrenaron tres clasificadores de spam con distintas distribuciones:
**Regresión Logística**, **Naive Bayes Bernoulli** y **KNN (k=5)**.

- **Logistic Regression** suele obtener un buen balance entre accuracy en entrenamiento y validación, siendo robusto ante datos de alta dimensión como histogramas de palabras.

- **Naive Bayes (Bernoulli)** es el modelo más adecuado conceptualmente para este problema, ya que asume variables binarias (presencia o ausencia de palabras). Su entrenamiento es muy rápido y generalmente generaliza bien, aunque puede ser superado por modelos más complejos.

- **KNN (k=5)** es un clasificador basado en distancia sin suposición de distribución. Tiende a sobreajustarse en datos de alta dimensión (2000 características), lo que puede reflejarse en un accuracy de entrenamiento muy alto pero menor en validación.

La **Precisión (Precision)** es especialmente relevante en detección de spam, ya que un falso positivo (clasificar un correo legítimo como spam) tiene un costo alto para el usuario. Un modelo con alta precisión garantiza que la mayoría de los correos marcados como spam realmente lo son.


<hr/>
<footer style="text-align:center; font-size:12px; color:gray;">
© 2026 UNAM Facultado de Ciencias – Todos los derechos reservados

</footer>